In [3]:
import re
import os
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob
import numpy as np
from numpy import *
import seaborn as sns


In [5]:
def max2(x):
    if len(x)==1: return x[0]
    x2 = x.copy()
    x2.sort()
    m1 = x2[-1]
    m2 = x2[-2]
    return (m2+m1)/2  

def get_on(trace):
        hmm_seq = "".join([str(string) for string in trace['HMMStates'].to_list()])
        sequence = hmm_seq
        #on state
        matches = re.findall(r'2+', sequence)
        lengths = [len(match) for match in matches]
        return lengths

def get_off(trace):
        hmm_seq = "".join([str(string) for string in trace['HMMStates'].to_list()])
        sequence = hmm_seq
        #on state
        matches = re.findall(r'1+', sequence)
        lengths = [len(match) for match in matches]
        return lengths

def get_on_intensity(trace):
        hmm_seq = "".join([str(string) for string in trace['HMMStates'].to_list()])
        sequence = hmm_seq
        intensity = trace['photon_number'].to_list()
        #off state
        background=[]
        matches = re.finditer(r'1+', sequence)
        for match in matches:
            start = match.start()
            end = match.end()
            background += intensity[start:end]
        #on state
        burst=[]
        matches = re.finditer(r'2+', sequence)
        for match in matches:
            start = match.start()
            end = match.end()
            period = intensity[start:end]
            m1m2 = max2(period)
            burst.append(m1m2-mean(background))
        return burst

In [7]:
data = pd.DataFrame(columns = ['filename','cellraw','gene','off','on','intensity','label','off-len','on-len','trace-len','pon'])
for i in glob('trace//*withBg_hmm2states.csv'):
    trace = pd.read_csv(i)
    on = get_on(trace)
    off = get_off(trace)
    on = [item*100/60 for item in on]
    off = [item*100/60 for item in off]    
    intensity = get_on_intensity(trace) 
    filename = i.split('\\')[-1]
    gene = 'FOS'
    cellraw = i.split('cellraw_')[-1].split('-dataAnalysis')[0]
    data.loc[len(data)] = filename,cellraw,gene,off,on,intensity,'onesite',np.sum(off),np.sum(on),len(trace)*100/60, np.sum(on)/(len(trace)*100/60)

In [9]:
data['on-mean'] = data['on'].apply(lambda x: np.mean(x))
data['off-mean'] = data['off'].apply(lambda x: np.mean(x))
data['intensity-mean'] = data['intensity'].apply(lambda x: np.mean(x))
data['bf'] = 1/(data['on-mean'] + data['off-mean'] )

In [11]:
data.to_csv('all_parameters.csv',index=None)